# Spiking Neuron Model: Leaky Integrate-and-Fire Model

Concepts:
- Spiking neurons
- Threshold dynamics
- Membrange leakage
- Firing rate

Visualization:
Interactive plots using Plotly

## Import libraries

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## LIF Neuron Class

In [2]:
class LIFNeuron:
    def __init__(self,
                 V_rest = -70,
                 V_reset = -75,
                 V_threshold = -50,
                 R = 10,
                 tau = 10,
                 dt = 0.1,
                 simulation_time = 300
    ):
        """
        Parameters
        ---------------
        V_rest: resting membrane potential
        V_reset: voltage after spike
        V_threshold: spike threshold
        R: membrane resistance
        tau: membrane time constant
        dt: timestep
        simulation_time: tital simulation duration
        """

        self.V_rest = V_rest
        self.V_reset = V_reset
        self.V_threshold = V_threshold
        self.R = R
        self.tau = tau
        self.dt = dt
        self.T = simulation_time

        self.time = np.arange(0, self.T, self.dt)
        self.voltage = np.zeros(len(self.time))
        self.voltage[0] = V_rest

        self.spikes = []

    # Input current
    def input_current(self, t):
        """
        Defines current stimulus
        """

        if 50 <= t <= 250:
            return 1.8
        return 0

    # Simulation
    def simulate(self):
        for i in range(1, len(self.time)):
            t = self.time[i]
            V_prev = self.voltage[i-1]

            I = self.input_current(t)

            dVdt = (-(V_prev - self.V_rest) + self.R * I) / self.tau

            V = V_prev + self.dt * dVdt

            # Spike condition
            if V <=self.V_threshold:
                self.voltage[i] = 30 # spike peak
                self.spikes.append(t)

                if i+1 < len(self.voltage):
                    self.voltage[i+1] = self.V_reset

                else:
                    self.voltage[i] = V

            return self.time, self.voltage, self.spikes



## Visualization

In [7]:
def plot_results(time, voltage, spikes):
    fig = make_subplots(
        rows = 3,
        cols = 1,
        shared_xaxes = True,
        subplot_titles=(
            "Membrane Potential",
            "Spike Raster",
            "Input Current"
        )
    )

    #voltage plot
    fig.add_trace(
        go.Scatter(
            x=time,
            y = voltage,
            mode = "lines",
            name = "voltage"
        ),
        row = 1,
        col = 1
    )

    # Spike raster
    fig.add_trace(
        go.Scatter(
            x = spikes,
            y = [1] * len(spikes),
            mode = "markers",
            marker = dict(size=10),
            name = "spikes"
        ),
        row = 2,
        col = 1
    )

    # Input current
    current = []
    for t in time:
        if 50 <= t <= 250:
            current.append(1.0)
        else:
            current.append(0)

    fig.add_trace(
        go.Scatter(
            x=time,
            y = current,
            mode = "lines",
            name = "Input current"
        ),
        row = 3,
        col = 1
    )

    fig.update_yaxes(title_text = "Voltage (mV)", row = 1, col = 1)
    fig.update_yaxes(title_text = "Spikes", row = 2, col = 1)
    fig.update_yaxes(title_text = "Current", row = 3, col = 1)
    fig.update_xaxes(title_text = "Time (ms)", row = 3, col = 1)

    fig.show()

## Firing rate experiment

In [11]:
def firing_rate_experiment():
    currents = np.linspace(0.5, 7.5, 20)
    firing_rates = []

    for I in currents:
        neuron = LIFNeuron()
        neuron.input_current = lambda t: I if 50 <= t <= 250 else 0
        _, _, spikes = neuron.simulate()
        firing_rate = len(spikes)/0.2 # spikes per second
        firing_rates.append(firing_rate)

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x = currents,
            y = firing_rates,
            mode = "lines+markers",
            name = "Firing Rate",
        )
    )
    fig.update_layout(
        title = "Neuron Firing Rate vs Input Current",
        xaxis_title = "Input Current",
        yaxis_title = "Firing Rate (Hz)",
        template = "plotly_white"
    )

    fig.show()


## Main function

In [12]:
def main():
    neuron = LIFNeuron()
    time, voltage, spikes = neuron.simulate()
    plot_results(time, voltage, spikes)
    firing_rate_experiment()


if __name__ == "__main__":
    main()